In [2]:
import pandas as pd

df = pd.read_csv("../../data/Clean_BNPParibas_Data.csv")

X = df.drop(columns=["contract_type", "customer_id"])
y = df["contract_type"]

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

Numeric features:
['age', 'tenure_months', 'monthly_charges', 'total_charges', 'support_tickets', 'churn']

Categorical features:
['internet_service', 'payment_method']


In [5]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.pipeline import Pipeline

adaboost_classifier = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            AdaBoostClassifier(
                random_state=42
            )
        )
    ]
)

adaboost_classifier.fit(
    X_train,
    y_train
)

adaboost_predictions = (
    adaboost_classifier.predict(X_test)
)

In [6]:
from sklearn.ensemble import ExtraTreesClassifier

extra_trees_classifier = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            ExtraTreesClassifier(
                n_estimators=100,
                random_state=42
            )
        )
    ]
)

extra_trees_classifier.fit(
    X_train,
    y_train
)

extra_trees_predictions = (
    extra_trees_classifier.predict(X_test)
)

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier

hist_gradient_classifier = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            HistGradientBoostingClassifier(
                random_state=42
            )
        )
    ]
)

hist_gradient_classifier.fit(
    X_train,
    y_train
)

hist_gradient_predictions = (
    hist_gradient_classifier.predict(X_test)
)

In [9]:
from sklearn.metrics import accuracy_score

classification_results = pd.DataFrame({
    "Model": [
        "Ada Boost",
        "Extra Tree",
        "Hist Gradient Boosting"
    ],
    "Accuracy": [
        accuracy_score(y_test, adaboost_predictions),
        accuracy_score(y_test, extra_trees_predictions),
        accuracy_score(y_test, hist_gradient_predictions)
    ]
})

display(
    classification_results.sort_values(
        "Accuracy",
        ascending=False
    )
)

,Model,Accuracy
0,Ada Boost,0.595628
2,Hist Gradient Boosting,0.568306
1,Extra Tree,0.513661
